# 03 - Build Processed Dataset

Este notebook LE o CSV bruto e ESCREVE o dataset analitico (`data/processed/loans_clean.parquet`).
Implementa `docs/cleaning_decisions.md` linha por linha. Se algo aqui divergir do documento,
o notebook para (assert) e sinaliza — nao decide sozinho.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

RAW_PATH = Path('..') / 'data' / 'raw' / 'accepted_2007_to_2018Q4.csv'
PROCESSED_DIR = Path('..') / 'data' / 'processed'
CHUNK_SIZE = 50000
CUTOFF_36 = pd.Timestamp('2015-12-01')
CUTOFF_60 = pd.Timestamp('2013-12-01')


## 1. Populacao aprovada

loan_status em {Fully Paid, Charged Off}; term 36m com issue_d ate dez/2015; term 60m ate
dez/2013. N esperado = 673.553. Se nao bater, o assert interrompe o notebook.

In [2]:
pop_chunks = []
n_total_file = 0
for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    n_total_file += len(chunk)
    status_ok = chunk['loan_status'].isin(['Fully Paid', 'Charged Off'])
    issue_dt = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    term_clean = chunk['term'].str.strip()
    is_36 = term_clean == '36 months'
    is_60 = term_clean == '60 months'
    mature = (is_36 & issue_dt.notna() & (issue_dt <= CUTOFF_36)) | (is_60 & issue_dt.notna() & (issue_dt <= CUTOFF_60))
    pop_mask = status_ok & mature
    if pop_mask.any():
        pop_chunks.append(chunk.loc[pop_mask].copy())

df = pd.concat(pop_chunks, ignore_index=True)
del pop_chunks

print(f'Linhas lidas do arquivo inteiro: {n_total_file:,}')
print(f'N da populacao: {len(df):,}')
assert len(df) == 673553, f'N inesperado: {len(df)} (esperado 673553) - PARANDO, diverge do contrato.'
print('N confere com o esperado (673.553). Prosseguindo.')


Linhas lidas do arquivo inteiro: 2,260,701
N da populacao: 673,553
N confere com o esperado (673.553). Prosseguindo.


## 2. Alvo

In [3]:
df['target'] = df['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1})
print('target criado. Distribuicao:')
print(df['target'].value_counts())
print(f'Taxa de default: {df["target"].mean() * 100:.4f}%')


target criado. Distribuicao:
target
0    573768
1     99785
Name: count, dtype: int64
Taxa de default: 14.8147%


## 3. Remocao das 5 linhas com dti > 100 (erro, nao outlier legitimo)

In [4]:
n_before = len(df)
df = df.loc[~(df['dti'] > 100)].reset_index(drop=True)
n_removed = n_before - len(df)
print(f'Linhas removidas (dti > 100): {n_removed} (esperado: 5)')
assert n_removed == 5, f'Esperava remover 5 linhas, removeu {n_removed} - PARANDO, diverge do contrato.'
print(f'Novo N: {len(df):,}')


Linhas removidas (dti > 100): 5 (esperado: 5)
Novo N: 673,548


## Transformacoes mecanicas (documentadas em cleaning_decisions.md)

Datas para datetime real; `term` para meses (numero, mantendo o nome — e EVAL_ONLY e precisa
ser numerico para o calculo de resultado financeiro); `emp_length` parseado para
`emp_length_anos` (coluna nova, a string original e descartada por redundancia); categoricas
de baixa cardinalidade com strip+lower (cardinalidade nao muda, ja documentado).

In [5]:
date_cols = ['issue_d', 'earliest_cr_line']
for c in date_cols:
    df[c] = pd.to_datetime(df[c], format='%b-%Y', errors='coerce')
print('Datas convertidas:', date_cols)

df['term'] = df['term'].str.strip().str.extract(r'(\d+)').astype(float)
print('term convertido para numerico (meses). Valores unicos:', sorted(df['term'].dropna().unique()))

def parse_emp_length(v):
    if pd.isna(v):
        return np.nan
    v = v.strip()
    if v == '< 1 year':
        return 0.0
    if v == '10+ years':
        return 10.0
    digits = ''.join(ch for ch in v if ch.isdigit())
    return float(digits) if digits else np.nan

df['emp_length_anos'] = df['emp_length'].apply(parse_emp_length)
print('emp_length_anos criado. Valores unicos:', sorted(df['emp_length_anos'].dropna().unique()))

low_card_cols = ['grade', 'sub_grade', 'home_ownership', 'purpose', 'verification_status',
                  'addr_state', 'initial_list_status', 'application_type']
for c in low_card_cols:
    before_card = df[c].nunique(dropna=True)
    df[c] = df[c].str.strip().str.lower()
    after_card = df[c].nunique(dropna=True)
    print(f'{c}: cardinalidade antes={before_card} depois={after_card}')


Datas convertidas: ['issue_d', 'earliest_cr_line']


term convertido para numerico (meses). Valores unicos: [np.float64(36.0), np.float64(60.0)]


emp_length_anos criado. Valores unicos: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0)]
grade: cardinalidade antes=7 depois=7
sub_grade: cardinalidade antes=35 depois=35
home_ownership: cardinalidade antes=6 depois=6


purpose: cardinalidade antes=14 depois=14
verification_status: cardinalidade antes=3 depois=3
addr_state: cardinalidade antes=51 depois=51
initial_list_status: cardinalidade antes=2 depois=2


application_type: cardinalidade antes=2 depois=2


## 4. Descarte de colunas (com motivo explicito)

Antes de descartar, a lista completa e impressa: coluna -> motivo.

In [6]:
drop_reasons = {}

for c in ['id', 'member_id', 'url']:
    drop_reasons[c] = 'Familia A - identificador'

family_B_original = [
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'pymnt_plan',
]
print(f'Familia B (classificacao original): {len(family_B_original)} colunas')
for c in family_B_original:
    drop_reasons[c] = 'Familia B - pos-originacao (leakage)'
drop_reasons['funded_amnt_inv'] = ('Familia B (REVISAR resolvido para B) - razao funded_amnt_inv/funded_amnt '
                                    'sobe de 0.235 (2007) a 0.9996 (2015): valor se completa apos a originacao')

for c in ['emp_title', 'title', 'desc', 'zip_code']:
    drop_reasons[c] = 'Familia D - texto livre'

zero_var_cols = ['policy_code', 'disbursement_method', 'pymnt_plan', 'hardship_flag', 'out_prncp', 'out_prncp_inv']
for c in zero_var_cols:
    if c in drop_reasons:
        drop_reasons[c] = drop_reasons[c] + ' + variancia zero na populacao'
    else:
        drop_reasons[c] = 'Variancia zero na populacao (cardinalidade = 1)'

block_dec2015 = ['open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
                  'total_bal_il', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
                  'inq_fi', 'total_cu_tl', 'inq_last_12m']
print(f'Bloco nascido dez/2015: {len(block_dec2015)} colunas')
for c in block_dec2015:
    drop_reasons[c] = 'Bloco nascido dez/2015 (~97.8% nulo na populacao, corte de maturidade termina antes do rollout completar)'

null100_cols = ['revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high',
                 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc',
                 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il',
                 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths',
                 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog',
                 'next_pymnt_d']
for c in null100_cols:
    if c in drop_reasons:
        drop_reasons[c] = drop_reasons[c] + ' + 100% nula na populacao'
    else:
        drop_reasons[c] = '100% nula na populacao (nasceu apos o fim da janela da populacao, mar/2017)'

drop_reasons['addr_state'] = ('Rare-category / alta cardinalidade (51 niveis, 1/3 sem amostra suficiente) - '
                                'dropada por decisao registrada em cleaning_decisions.md')

drop_reasons['emp_length'] = 'Transformacao mecanica - substituida por emp_length_anos (parseada)'

print()
print(f'Total de colunas a descartar: {len(drop_reasons)}')
for c, reason in sorted(drop_reasons.items()):
    print(f'  {c}: {reason}')


Familia B (classificacao original): 37 colunas
Bloco nascido dez/2015: 13 colunas

Total de colunas a descartar: 75
  addr_state: Rare-category / alta cardinalidade (51 niveis, 1/3 sem amostra suficiente) - dropada por decisao registrada em cleaning_decisions.md
  all_util: Bloco nascido dez/2015 (~97.8% nulo na populacao, corte de maturidade termina antes do rollout completar)
  collection_recovery_fee: Familia B - pos-originacao (leakage)
  debt_settlement_flag: Familia B - pos-originacao (leakage)
  debt_settlement_flag_date: Familia B - pos-originacao (leakage)
  deferral_term: Familia B - pos-originacao (leakage)
  desc: Familia D - texto livre
  disbursement_method: Variancia zero na populacao (cardinalidade = 1)
  emp_length: Transformacao mecanica - substituida por emp_length_anos (parseada)
  emp_title: Familia D - texto livre
  funded_amnt_inv: Familia B (REVISAR resolvido para B) - razao funded_amnt_inv/funded_amnt sobe de 0.235 (2007) a 0.9996 (2015): valor se completa apos

In [7]:
cols_to_drop = [c for c in drop_reasons if c in df.columns]
missing = [c for c in drop_reasons if c not in df.columns]
if missing:
    print(f'ATENCAO: colunas na lista de descarte que nao existem no dataframe: {missing}')

shape_before = df.shape
df = df.drop(columns=cols_to_drop)
print(f'Shape antes: {shape_before} -> depois: {df.shape}')


Shape antes: (673548, 153) -> depois: (673548, 78)


## 5. Familia E (EVAL_ONLY) e exclusoes provisorias

Familia E nunca entra em nenhum feature set, em nenhuma fase. int_rate, grade e sub_grade
sao mantidas no dataset mas marcadas como exclusao provisoria (ver "Deferred to modeling"
em cleaning_decisions.md) — a serem resolvidas na modelagem, com versao comparativa.

In [8]:
EVAL_ONLY = ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
print('EVAL_ONLY (nunca features, em nenhuma fase):', EVAL_ONLY)
print('Todas presentes no dataframe?', all(c in df.columns for c in EVAL_ONLY))

PROVISIONAL_EXCLUDE = ['int_rate', 'grade', 'sub_grade']
print()
print('PROVISIONAL_EXCLUDE (mantidas no dataset, excluidas do feature set ate a modelagem')
print('comparativa decidir se entram):', PROVISIONAL_EXCLUDE)
print('Todas presentes no dataframe?', all(c in df.columns for c in PROVISIONAL_EXCLUDE))


EVAL_ONLY (nunca features, em nenhuma fase): ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
Todas presentes no dataframe? True

PROVISIONAL_EXCLUDE (mantidas no dataset, excluidas do feature set ate a modelagem
comparativa decidir se entram): ['int_rate', 'grade', 'sub_grade']
Todas presentes no dataframe? True


## 6. MNAR — flag + sentinela 999 (mths_since_*) e -1 (emp_length_anos)

In [9]:
mnar_cols_999 = ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_recent_bc_dlq',
                  'mths_since_recent_revol_delinq', 'mths_since_last_major_derog']

for c in mnar_cols_999:
    pct_null_before = df[c].isnull().mean() * 100
    flag_col = f'{c}_missing'
    df[flag_col] = df[c].isnull().astype(int)
    df[c] = df[c].fillna(999)
    pct_null_after = df[c].isnull().mean() * 100
    print(f'{c}: %nulo antes={pct_null_before:.2f}% depois={pct_null_after:.2f}% | {flag_col} soma={df[flag_col].sum():,}')

print()
pct_null_before = df['emp_length_anos'].isnull().mean() * 100
df['emp_length_missing'] = df['emp_length_anos'].isnull().astype(int)
df['emp_length_anos'] = df['emp_length_anos'].fillna(-1)
pct_null_after = df['emp_length_anos'].isnull().mean() * 100
print(f'emp_length_anos: %nulo antes={pct_null_before:.2f}% depois={pct_null_after:.2f}% | '
      f'emp_length_missing soma={df["emp_length_missing"].sum():,}')


mths_since_last_delinq: %nulo antes=51.77% depois=0.00% | mths_since_last_delinq_missing soma=348,666


mths_since_last_record: %nulo antes=84.58% depois=0.00% | mths_since_last_record_missing soma=569,678
mths_since_recent_bc_dlq: %nulo antes=77.07% depois=0.00% | mths_since_recent_bc_dlq_missing soma=519,104
mths_since_recent_revol_delinq: %nulo antes=67.76% depois=0.00% | mths_since_recent_revol_delinq_missing soma=456,366
mths_since_last_major_derog: %nulo antes=75.42% depois=0.00% | mths_since_last_major_derog_missing soma=508,015

emp_length_anos: %nulo antes=5.58% depois=0.00% | emp_length_missing soma=37,567


## 7. Bloco de bureau pre-2012 — flag era_pre_2012 + sentinela -1

Colunas identificadas programaticamente pela sobreposicao com a mascara
`tot_cur_bal.isna()` (ja validada em 02_cleaning.ipynb), nao por uma lista fixa.

In [10]:
ref_mask = df['tot_cur_bal'].isna()
already_handled = set(EVAL_ONLY) | set(mnar_cols_999) | {'emp_length_anos', 'emp_length_missing', 'target'}
candidate_cols = [c for c in df.columns if c not in already_handled and df[c].isnull().sum() > 0]

bureau_2012_cols = []
for c in candidate_cols:
    col_mask = df[c].isna()
    if col_mask.sum() == 0:
        continue
    overlap_of_ref = (col_mask & ref_mask).sum() / ref_mask.sum() * 100 if ref_mask.sum() > 0 else 0
    overlap_of_col = (col_mask & ref_mask).sum() / col_mask.sum() * 100 if col_mask.sum() > 0 else 0
    if overlap_of_ref > 95 and overlap_of_col > 95:
        bureau_2012_cols.append(c)

print(f'Colunas do bloco bureau 2012 identificadas via mascara de tot_cur_bal: {len(bureau_2012_cols)}')
print(bureau_2012_cols)
print('(nota: o contrato estimava "~24"; a contagem exata medida via sobreposicao de mascara prevalece)')


Colunas do bloco bureau 2012 identificadas via mascara de tot_cur_bal: 21
['tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'avg_cur_bal', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'tot_hi_cred_lim', 'total_il_high_credit_limit']
(nota: o contrato estimava "~24"; a contagem exata medida via sobreposicao de mascara prevalece)


In [11]:
df['era_pre_2012'] = ref_mask.astype(int)
for c in bureau_2012_cols:
    df[c] = df[c].fillna(-1)

print(f'era_pre_2012 - soma da flag: {df["era_pre_2012"].sum():,}')
print(f'%_populacao com era_pre_2012=1: {df["era_pre_2012"].mean() * 100:.4f}%')
print('Compara com a sobreposicao ja medida em 02_cleaning.ipynb (~10.0255% / 67.527 linhas).')


era_pre_2012 - soma da flag: 67,527
%_populacao com era_pre_2012=1: 10.0256%
Compara com a sobreposicao ja medida em 02_cleaning.ipynb (~10.0255% / 67.527 linhas).


## 8. Consolidacao de categorias raras

In [12]:
print('home_ownership - antes:')
print(df['home_ownership'].value_counts())

df['home_ownership'] = df['home_ownership'].replace({'any': 'other', 'none': 'other'})

print()
print('home_ownership - depois:')
print(df['home_ownership'].value_counts())


home_ownership - antes:
home_ownership
mortgage    320795
rent        285754
own          66808
other          144
none            45
any              2
Name: count, dtype: int64

home_ownership - depois:
home_ownership
mortgage    320795
rent        285754
own          66808
other          191
Name: count, dtype: int64


In [13]:
print('purpose - antes:')
print(df['purpose'].value_counts())

df['purpose'] = df['purpose'].replace({'wedding': 'other', 'educational': 'other', 'renewable_energy': 'other'})

print()
print('purpose - depois:')
print(df['purpose'].value_counts())


purpose - antes:
purpose
debt_consolidation    385923
credit_card           159055
home_improvement       39094
other                  36375
major_purchase         14076
small_business          8635
car                     7786
medical                 7245
moving                  4815
vacation                4426
house                   2990
wedding                 2290
renewable_energy         512
educational              326
Name: count, dtype: int64



purpose - depois:
purpose
debt_consolidation    385923
credit_card           159055
other                  39503
home_improvement       39094
major_purchase         14076
small_business          8635
car                     7786
medical                 7245
moving                  4815
vacation                4426
house                   2990
Name: count, dtype: int64


## 9. Nulos remanescentes (sem imputar — so reportar)

In [14]:
remaining_nulls = (df.isnull().mean() * 100)
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print(f'Colunas com nulo remanescente: {len(remaining_nulls)}')
remaining_nulls.to_frame('%_nulo')


Colunas com nulo remanescente: 23


,%_nulo
dti_joint,99.965407
annual_inc_joint,99.965259
verification_status_joint,99.965259
il_util,98.097389
mths_since_recent_inq,16.971025
mo_sin_old_il_acct,13.294821
num_tl_120dpd_2m,13.134476
num_sats,8.290575
num_bc_sats,8.290575
bc_util,7.989631


## 10. Escrita do dataset analitico

In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = PROCESSED_DIR / 'loans_clean.parquet'
df.to_parquet(parquet_path, index=False)

sample_path = PROCESSED_DIR / 'loans_clean_sample.csv'
df.sample(n=1000, random_state=42).to_csv(sample_path, index=False)

parquet_size_mb = parquet_path.stat().st_size / 1024**2
sample_size_kb = sample_path.stat().st_size / 1024

print(f'Shape final: {df.shape}')
print(f'Parquet: {parquet_path} - {parquet_size_mb:.2f} MB')
print(f'Sample CSV: {sample_path} - {sample_size_kb:.2f} KB')
print(f'Taxa de default final: {df["target"].mean() * 100:.4f}%')


Shape final: (673548, 85)
Parquet: ..\data\processed\loans_clean.parquet - 45.53 MB
Sample CSV: ..\data\processed\loans_clean_sample.csv - 428.12 KB
Taxa de default final: 14.8148%


## 11. Verificacao de integridade final

In [16]:
family_B_all = family_B_original + ['funded_amnt_inv']
survived_B = [c for c in family_B_all if c in df.columns]
print(f'Colunas familia B sobreviventes (esperado 0): {survived_B}')
assert len(survived_B) == 0, 'PARANDO: coluna(s) da familia B sobreviveram ao descarte.'

print()
print('EVAL_ONLY presentes no dataframe:', [c for c in EVAL_ONLY if c in df.columns])
print('EVAL_ONLY faltando:', [c for c in EVAL_ONLY if c not in df.columns])
print('(EVAL_ONLY existe no arquivo para calculo de resultado financeiro e auditoria; a')
print(' exclusao do feature set e responsabilidade da fase de modelagem, nao deste notebook.)')


Colunas familia B sobreviventes (esperado 0): []

EVAL_ONLY presentes no dataframe: ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
EVAL_ONLY faltando: []
(EVAL_ONLY existe no arquivo para calculo de resultado financeiro e auditoria; a
 exclusao do feature set e responsabilidade da fase de modelagem, nao deste notebook.)


In [17]:
id_status = 'id foi removida do dataset (familia A - identificador), nao ha o que checar.' if 'id' not in df.columns else str(df['id'].dtype)
print('id:', id_status)
print()
print('dtypes finais:')
print(df.dtypes.to_string())


id: id foi removida do dataset (familia A - identificador), nao ha o que checar.

dtypes finais:
loan_amnt                                        float64
funded_amnt                                      float64
term                                             float64
int_rate                                         float64
installment                                      float64
grade                                                str
sub_grade                                            str
home_ownership                                       str
annual_inc                                       float64
verification_status                                  str
issue_d                                   datetime64[us]
loan_status                                          str
purpose                                              str
dti                                              float64
delinq_2yrs                                      float64
earliest_cr_line                          dateti

In [18]:
expected_object_cols = {'loan_status', 'grade', 'sub_grade', 'home_ownership', 'verification_status',
                          'purpose', 'initial_list_status', 'application_type', 'verification_status_joint'}
unexpected_object = [c for c in df.columns if df[c].dtype == 'object' and c not in expected_object_cols]
print(f'Colunas object nao esperadas (deveriam ser numericas): {unexpected_object}')
if unexpected_object:
    print('ATENCAO: revisar antes de seguir.')
else:
    print('Nenhuma coluna object inesperada. Todas as colunas categoricas remanescentes sao as esperadas.')


Colunas object nao esperadas (deveriam ser numericas): []
Nenhuma coluna object inesperada. Todas as colunas categoricas remanescentes sao as esperadas.
